# 06 — מטריצת העמדות: 14 מפלגות × 25 שאלות

**המטרה:** לאחד את התיוגים של המצעים (03) ושל הצעות החוק (05) לטבלה אחת של מספרים, שממנה ייבנו וקטורי המפלגות.

**ההחלטות שכבר קיבלנו:**
- כל מפלגה מגיעה ממקור אחד בלבד: מצע או הצעות חוק. אין חפיפה בין השכבות.
- `sure` → ‎±1, ‏`partial` → ‎±0.5, ‏`unclear` → ‏0, ‏`not_covered` → ‏NaN.
- **0 ו-NaN הם לא אותו דבר.** 0 הוא מידע: המפלגה עוסקת בנושא בלי כיוון. NaN אומר שאין לנו מידע.

אחרי כל משימה יש תא בדיקה. אם הוא רץ בלי שגיאה, אפשר להמשיך.

In [21]:
import glob, json
import numpy as np
import pandas as pd

questions = pd.DataFrame(json.load(open("../data/seed/quiz.json", encoding="utf-8"))["questions"])
parties = json.load(open("../data/seed/parties.json", encoding="utf-8"))
parties = parties if isinstance(parties, list) else parties["parties"]
SLUGS = [p["slug"] for p in parties]
QIDS = sorted(questions["id"])
len(SLUGS), len(QIDS)

(14, 25)

## משימה 1 — לטעון את כל התיוגים לטבלה אחת

קבצי `platform_coverage_*.csv` וקבצי `bill_coverage_*.csv`, כולם ל-DataFrame אחד בשם `labels`.

- הוסף עמודה `source` עם הערך `'platform'` או `'bill'`, כדי שבהמשך נדע מאיפה הגיעה כל עמדה.
- לשני סוגי הקבצים יש עמודות שונות (`passage`/`quote` מול `bill`/`title`/...). זה בסדר: `pd.concat` ימלא NaN במקומות החסרים.

**רמז:** `glob.glob`, ‏`pd.read_csv`, ‏`.assign(source=...)`, ‏`pd.concat`. ראית את זה במחברת 04.

In [22]:
# TODO

plat = glob.glob('platform_coverage_*.csv')
platforms = [pd.read_csv(f).assign(source="platform") for f in plat]
bill = glob.glob('bill_coverage_*.csv')
bills = [pd.read_csv(b).assign(source="bill") for b in bill]
labels = pd.concat(platforms + bills, ignore_index=True)

labels["priority"] = labels["source"].map({"platform": 0, "bill": 1}) 
labels["no_info"] = labels["confidence"] == "not_covered"
labels = labels.sort_values(["no_info", "priority"])           
labels = labels.drop_duplicates(["party", "question_id"])

In [23]:
# בדיקה
assert labels.groupby("party").size().eq(25).all(), "לכל מפלגה צריכות להיות בדיוק 25 שורות"
assert not labels.duplicated(["party", "question_id"]).any(), "אותה מפלגה ואותה שאלה מופיעות פעמיים"
assert set(labels["source"]) == {"platform", "bill"}
print(len(labels), "שורות,", labels["party"].nunique(), "מפלגות")

350 שורות, 14 מפלגות


## משימה 2 — ליישר את שמות המפלגות

בשמות הקבצים יש שלוש מפלגות שהשם שלהן שונה מה-`slug` שלהן ב-`parties.json`. מצא אותן בעצמך: הקוד למטה מראה אילו שמות לא תואמים. אחר כך תקן אותן בעמודה `party`.

**למה זה חשוב:** בשלב 4 הווקטורים ייכתבו לטבלה `PartyVector` לפי ה-slug. מפלגה עם שם לא נכון פשוט לא תקבל וקטור, ואף שגיאה לא תתריע על זה.

**רמז:** מילון ו-`.replace`.

In [24]:
print("בקבצים ולא ב-parties.json:", sorted(set(labels["party"]) - set(SLUGS)))
print("ב-parties.json ולא בקבצים:", sorted(set(SLUGS) - set(labels["party"])))

בקבצים ולא ב-parties.json: ['democrats', 'otzma', 'yachad']
ב-parties.json ולא בקבצים: ['beyachad', 'otzma-yehudit', 'the-democrats']


In [25]:
# TODO
rename = {"democrats": "the-democrats", "otzma": "otzma-yehudit", "yachad": "beyachad"}
labels["party"] = labels["party"].replace(rename)

In [26]:
# בדיקה
assert set(labels["party"]) == set(SLUGS), sorted(set(labels["party"]) ^ set(SLUGS))

## משימה 3 — מתיוג למספר

צור עמודה `value` לפי הכללים שבראש המחברת.

**שים לב למלכודת:** ב-`not_covered` העמודה `stance` ריקה. ב-`unclear` היא `0`. ודא שאחרי ההמרה הראשון יוצא NaN והשני 0, ולא להפך, ולא שניהם 0.

**רמז:** מילון מ-`confidence` למשקל, ‏`.map`, ואז כפל ב-`stance`. מה קורה כשכופלים NaN במספר?

In [27]:
# TODO
conf_to_weight = {"sure":1,"partial":0.5,"unclear":0,"not_covered":0}
labels["value"] = labels["confidence"].map(conf_to_weight) * labels["stance"]

In [28]:
# בדיקה
allowed = {-1.0, -0.5, 0.0, 0.5, 1.0}
assert set(labels["value"].dropna()) <= allowed, set(labels["value"].dropna()) - allowed
assert labels["value"].isna().sum() == (labels["confidence"] == "not_covered").sum(), "NaN רק איפה שאין מידע"
assert (labels.loc[labels["confidence"] == "unclear", "value"] == 0).all(), "unclear צריך להיות 0, לא NaN"
labels["value"].value_counts(dropna=False)

value
 NaN    202
 0.5     47
 1.0     44
 0.0     26
-0.5     17
-1.0     14
Name: count, dtype: int64

## משימה 4 — המטריצה

הפוך את `labels` למטריצה `M`: שורה לכל מפלגה, עמודה לכל שאלה, וערכים מתוך `value`.

אחר כך צור טבלת כיסוי `coverage`, עם שורה לכל מפלגה ושלוש עמודות:
- `source`: מאיזה מקור הגיעו העמדות
- `covered`: כמה שאלות הן לא NaN
- `directional`: כמה מהן הן לא 0

מיין לפי `covered`, מהנמוך לגבוה.

**רמז:** `pivot`, כמו במחברת 04. בשביל הספירות: `notna()` ו-`ne(0)` על המטריצה. חשוב על מה ש-`ne(0)` מחזיר עבור NaN.

In [29]:
# TODO
M = labels.pivot(index="party", columns="question_id", values="value")
coverage = pd.DataFrame({
    "source": labels.groupby("party")["source"].first(),
    "covered": M.notna().sum(axis=1),
    "directional": (M.notna() & M.ne(0)).sum(axis=1),
}).sort_values("covered")


In [30]:
# בדיקה
assert M.shape == (14, 25), M.shape
assert list(M.columns) == QIDS
assert (coverage["directional"] <= coverage["covered"]).all()
assert coverage["covered"].sum() == M.notna().sum().sum()
coverage

,source,covered,directional
party,,,
raam,bill,4,4
joint-list,platform,5,5
kachol-lavan,bill,5,5
religious-zionism,bill,7,5
yahadut-hatorah,bill,9,8
beyachad,platform,10,7
zionist-home,platform,10,6
amcha-israel,platform,12,10
likud,bill,12,10


## משימה 5 — לשמור

שמור את `M` ל-`party_positions.json` במבנה הזה:

```json
{"likud": {"q01": -0.5, "q02": 1.0, ...}, ...}
```

**בלי NaN:** שאלה שאין עליה מידע פשוט לא מופיעה. זה הפורמט שבשלב 4 נעביר ל-`PartyVector.vectorJson`. בקובץ JSON תקני אין `NaN`, ו-`json.dump` יכתוב אותו בלי להתלונן, וכך ייצור קובץ שבור.

**רמז:** לולאה על `M.iterrows()` ו-`dropna()` על כל שורה. ‏`ensure_ascii=False`, ‏`indent=1`.

In [31]:
# TODO
positions = {}
for party, row in M.iterrows():
    positions[party] = row.dropna().to_dict()         # שלב 2

with open("party_positions.json", "w", encoding="utf-8") as f:
    json.dump(positions, f, ensure_ascii=False, indent=1)

In [32]:
# בדיקה
saved = json.load(open("party_positions.json", encoding="utf-8"))
assert set(saved) == set(SLUGS)
assert sum(len(v) for v in saved.values()) == M.notna().sum().sum()
assert "NaN" not in open("party_positions.json", encoding="utf-8").read()
print("נשמר")

נשמר


## לפני שממשיכים לשלב ב — שאלות לחשיבה (בלי קוד)

1. לאיזו מפלגה יש הכי מעט עמדות? לאיזו שאלה יש הכי מעט מפלגות?
2. סמן לכל מפלגה את הגוש שלה (`bloc` ב-`parties.json`). האם החוסר מתפזר באופן שווה בין הגושים, או שהוא מתרכז בצד אחד? אם הוא מתרכז, איך זה עלול להטות את ההתאמה?
3. בכמה שאלות **כל** 14 המפלגות מכוסות? מה זה אומר על קוסינוס מוסווה בין משתמש למפלגה עם 4 עמדות?

על שאלה 3 נבנה את שלב ב.

In [33]:
coverage
M


question_id,q01,q02,q03,q04,q05,q06,q07,q08,q09,q10,...,q16,q17,q18,q19,q20,q21,q22,q23,q24,q25
party,,,,,,,,,,,,,,,,,,,,,
amcha-israel,-1.0,0.5,NaN,NaN,0.5,0.5,-0.5,NaN,NaN,NaN,...,0.0,NaN,1.0,NaN,1.0,0.0,0.5,NaN,NaN,NaN
beyachad,NaN,NaN,NaN,0.5,0.0,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN
joint-list,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,1.0,-0.5,NaN,NaN,NaN
kachol-lavan,NaN,NaN,NaN,NaN,NaN,NaN,0.5,NaN,NaN,0.5,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.5,NaN
likud,-0.5,1.0,NaN,NaN,0.5,0.5,0.5,NaN,0.5,0.5,...,NaN,0.5,0.5,NaN,NaN,NaN,NaN,NaN,1.0,0.0
otzma-yehudit,-0.5,0.5,NaN,NaN,0.5,0.5,0.5,NaN,NaN,NaN,...,NaN,NaN,0.5,NaN,0.0,0.0,0.5,NaN,NaN,1.0
raam,NaN,NaN,NaN,NaN,NaN,0.5,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN
religious-zionism,-0.5,1.0,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.5,NaN,NaN,0.0,NaN,NaN,1.0,NaN
shas,-0.5,1.0,NaN,NaN,NaN,-0.5,1.0,0.0,1.0,1.0,...,NaN,NaN,0.5,NaN,NaN,0.0,NaN,NaN,0.5,NaN
